In [0]:
path = "/Volumes/spark_definitive_guide/spark_book/simple-ml/"
df = spark.read.json(path)
df.orderBy("value2").show()

# RFormula

prepare data to use in ML models

In [0]:
from pyspark.ml.feature import RFormula 

supervised = RFormula(formula="lab ~ . + color:value1 + color:value2")


In [0]:
fittedRF = supervised.fit(df)
print(fittedRF)

In [0]:
preparedDF = fittedRF.transform(df)
preparedDF.show()

# Split data

In [0]:
train, test = preparedDF.randomSplit([0.7, 0.3])
print(train.count())
print(test.count())

# Logistic Regression

Logistic Regression - import

In [0]:
from pyspark.ml.classification import LogisticRegression
lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=10)

Parameters

In [0]:
print(lr.explainParams())

LogisticRegressionModel

In [0]:
fittedLR = lr.fit(train)

Prediction

In [0]:
fittedLR.transform(train).select("label", "prediction").show()

# Pipelines

In [0]:
train, test = df.randomSplit([0.7, 0.3])

RFormula and Logistic Regression

In [0]:
rForm = RFormula()
lr = LogisticRegression().setLabelCol("label").setFeaturesCol("features")

Pipeline stage

In [0]:
from pyspark.ml import Pipeline

stage = [rForm, lr]
pipeline = Pipeline().setStages(stage)

# Train with different hyperparameters

In [0]:
from pyspark.ml.tuning import ParamGridBuilder

params = ParamGridBuilder()\
    .addGrid(rForm.formula, [
        "lab ~ . + color:value1",
        "lab ~ . + color:value1 + color:value2"])\
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])\
    .addGrid(lr.regParam, [0.1, 2.0])\
    .build()



# Evaluator

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator()\
    .setMetricName("areaUnderROC")\
    .setRawPredictionCol("prediction")\
    .setLabelCol("label")


# TrainValidationSplit

In [0]:
from pyspark.ml.tuning import TrainValidationSplit

# grid parameter (param) \ model (pipeline) \ evaluator (evaluator)
tvs = TrainValidationSplit()\
    .setTrainRatio(0.75)\
    .setEstimatorParamMaps(params)  \
    .setEstimator(pipeline)  \
    .setEvaluator(evaluator) #
    

# Fit / Transform

In [0]:
import os
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/spark_definitive_guide/spark_book/simple-ml"
tvsFitted = tvs.fit(train)

In [0]:
evaluator.evaluate(tvsFitted.transform(test))

# Training Summary

In [0]:
# training summary — best model - TrainValidationSplit
best = tvsFitted.bestModel          # PipelineModel: RFormula + LogisticRegression
rform_best, lr_best = best.stages

print("Best RFormula formula:", rform_best.getFormula())
print("ElasticNetParam:", lr_best.getElasticNetParam(), "| RegParam:", lr_best.getRegParam())

# Metrics best model
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

pred_train = best.transform(train)

print("\n--- Training summary (best model applied to train) ---")
print("Accuracy:",
      MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")\
          .evaluate(pred_train))
print("AreaUnderROC:",
      BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prediction", metricName="areaUnderROC")\
          .evaluate(pred_train))

print("\nConfusion matrix (label x prediction):")
display(pred_train.groupBy("label", "prediction").count())

print("\n--- Validation metrics for all grid combinations ---")
display(tvsFitted.validationMetrics)

# Save model on volume

In [0]:
# Persist model on volume
model_path = "/Volumes/spark_definitive_guide/spark_book/simple-ml/models/tvs-model"
tvsFitted.write().overwrite().save(model_path)

# Load model from volume

In [0]:
from pyspark.ml.tuning import TrainValidationSplitModel

tvs_loaded = TrainValidationSplitModel.load(model_path)
print("Model reloaded. Best formula:", tvs_loaded.bestModel.stages[0].getFormula())

print("\nPredictions recalculated by the reloaded model on test:")
tvs_loaded.transform(test).select("lab", "label", "prediction").show(5)

# MLFlow

In [0]:
# MLflow — log the training as a "run": parameters, metrics, and the model as an artifact
# Note: mlflow.spark.log_model requires a classic cluster (not available in free edition/serverless);
# so we log the model already saved on the volume as the run's artifact.
import mlflow

mlflow.set_experiment("/Users/bruno.lunardi.92@gmail.com/mlib-test")

with mlflow.start_run() as run:
    mlflow.log_params({
        "formula": rform_best.getFormula(),
        "regParam": lr_best.getRegParam(),
        "elasticNetParam": lr_best.getElasticNetParam(),
    })
    mlflow.log_metric("train_accuracy",
        MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")\
            .evaluate(pred_train))
    mlflow.log_metric("test_auc", evaluator.evaluate(tvsFitted.transform(test)))
    mlflow.log_artifacts(model_path, artifact_path="model")  # copies the model from the volume into the run
    print("Run created:", run.info.run_id)
    print("Artifacts at:", run.info.artifact_uri)